# Module 09: Endogeneity & Instrumental Variables
*Statistics for Econometrics — An intermediate course for researchers*

This module covers endogeneity problems in regression, the instrumental variables (IV) approach to causal inference, two-stage least squares (2SLS), testing instrument validity, and practical applications to real-world causal questions.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

try:
    import statsmodels.api as sm
    from statsmodels.formula.api import ols
    from statsmodels.stats.diagnostic import linear_rainbow
except ImportError:
    !pip install statsmodels
    import statsmodels.api as sm
    from statsmodels.formula.api import ols
    from statsmodels.stats.diagnostic import linear_rainbow

try:
    from linearmodels.iv import IV2SLS
except ImportError:
    !pip install linearmodels
    from linearmodels.iv import IV2SLS

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
np.random.seed(42)

## 9.1 — The Endogeneity Problem

**Endogeneity** occurs when a regressor is correlated with the error term, violating the fundamental OLS assumption. This can arise from:

1. **Omitted variable bias (OVB)**: A relevant variable is left out of the model and correlates with included regressors
2. **Simultaneity/reverse causality**: The outcome affects the regressor (feedback loop)
3. **Measurement error**: The regressor is measured with error

When endogeneity is present, OLS estimators are **biased and inconsistent**, even in large samples.

In [ ]:
# Simulate omitted variable bias (OVB)
# True model: Y = 2 + 3*X1 + 4*X2 + ε
# But X1 and X2 are correlated (corr = 0.6)

n = 500
np.random.seed(42)

# Create correlated X1 and X2
X2 = np.random.normal(0, 1, n)
X1 = 0.6 * X2 + np.sqrt(1 - 0.6**2) * np.random.normal(0, 1, n)

# Generate Y from the true model: Y = 2 + 3*X1 + 4*X2 + ε
epsilon = np.random.normal(0, 1, n)
Y = 2 + 3*X1 + 4*X2 + epsilon

data_ovb = pd.DataFrame({'Y': Y, 'X1': X1, 'X2': X2})

# Model 1: OLS with both X1 and X2 (correct specification)
model_correct = ols('Y ~ X1 + X2', data=data_ovb).fit()

# Model 2: OLS with X1 only (omitting X2 - biased)
model_omitted = ols('Y ~ X1', data=data_ovb).fit()

print("="*70)
print("OMITTED VARIABLE BIAS (OVB) DEMONSTRATION")
print("="*70)
print(f"\nTrue value of β₁ (coefficient on X1): 3.0")
print(f"\nOLS WITH BOTH X1 and X2 (correct model):")
print(f"  β̂₁ = {model_correct.params['X1']:.4f}")
print(f"  SE = {model_correct.bse['X1']:.4f}")
print(f"  ✓ Unbiased estimate")

print(f"\nOLS WITH X1 ONLY (omitting X2):")
print(f"  β̂₁ = {model_omitted.params['X1']:.4f}")
print(f"  SE = {model_omitted.bse['X1']:.4f}")
print(f"  ✗ BIASED estimate (bias = {model_omitted.params['X1'] - 3:.4f})")

print(f"\nInterpretation:")
print(f"  X2 has coefficient 4 and corr(X1,X2) = 0.6")
print(f"  Expected bias ≈ 4 × 0.6 = 2.4")
print(f"  Actual bias ≈ {model_omitted.params['X1'] - 3:.4f}")

## 9.2 — Instrumental Variables: The Idea

An **instrumental variable (IV)** Z is a variable that:
1. **Affects the outcome only through the endogenous regressor** (exclusion restriction): Z → X → Y
2. **Is correlated with the endogenous regressor** (relevance): Cov(Z, X) ≠ 0
3. **Is uncorrelated with the error term** (exogeneity): Cov(Z, ε) = 0

The IV estimator exploits this structure to estimate the causal effect of X on Y.

In [ ]:
# Visualize the IV structure
# Create a simple DAG-like illustration

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel 1: OVB problem
ax = axes[0]
ax.text(0.2, 0.8, 'U\n(confounder)', fontsize=12, ha='center', 
        bbox=dict(boxstyle='round', facecolor='red', alpha=0.3))
ax.text(0.5, 0.8, 'X\n(endogenous)', fontsize=12, ha='center',
        bbox=dict(boxstyle='round', facecolor='orange', alpha=0.3))
ax.text(0.8, 0.8, 'Y\n(outcome)', fontsize=12, ha='center',
        bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.3))
ax.arrow(0.25, 0.75, 0.2, 0, head_width=0.03, head_length=0.05, fc='red', ec='red')
ax.arrow(0.25, 0.72, 0.2, 0, head_width=0.03, head_length=0.05, fc='red', ec='red')
ax.arrow(0.55, 0.75, 0.2, 0, head_width=0.03, head_length=0.05, fc='black', ec='black')
ax.text(0.35, 0.78, 'corr', fontsize=9, color='red')
ax.text(0.65, 0.78, 'causal', fontsize=9)
ax.set_xlim(0, 1)
ax.set_ylim(0.6, 1)
ax.axis('off')
ax.set_title('Problem: Omitted Variable Bias', fontsize=13, fontweight='bold')

# Panel 2: IV solution
ax = axes[1]
ax.text(0.2, 0.8, 'Z\n(instrument)', fontsize=12, ha='center',
        bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.3))
ax.text(0.5, 0.8, 'X\n(endogenous)', fontsize=12, ha='center',
        bbox=dict(boxstyle='round', facecolor='orange', alpha=0.3))
ax.text(0.8, 0.8, 'Y\n(outcome)', fontsize=12, ha='center',
        bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.3))
ax.arrow(0.28, 0.75, 0.17, 0, head_width=0.03, head_length=0.05, fc='green', ec='green')
ax.arrow(0.55, 0.75, 0.2, 0, head_width=0.03, head_length=0.05, fc='black', ec='black')
ax.text(0.37, 0.78, 'relevant', fontsize=9, color='green')
ax.text(0.65, 0.78, 'causal', fontsize=9)
ax.set_xlim(0, 1)
ax.set_ylim(0.6, 1)
ax.axis('off')
ax.set_title('Solution: Instrumental Variable', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

print("The IV breaks the endogeneity by providing exogenous variation in X.")

## 9.3 — The Simple IV Estimator

With one endogenous regressor and one instrument, the **IV estimator** is:

β̂₁^IV = Cov(Z, Y) / Cov(Z, X)

This exploits the correlation between Z and Y (via X) while using Z's exogeneity.

In [ ]:
# Simulate endogeneity problem with IV solution
# Model: Y = 3 + 2*X + U + ε
# Confounder U affects both X and Y
# Instrument Z affects X but is exogenous (uncorrelated with U and ε)

n = 500
np.random.seed(42)

# Exogenous instrument
Z = np.random.normal(0, 1, n)

# Confounder (unobserved)
U = np.random.normal(0, 1, n)

# Endogenous regressor: X = 1 + 0.5*Z + 0.8*U + noise
# Z affects X (relevance, π₁ = 0.5)
# U affects X (creates endogeneity)
X = 1 + 0.5*Z + 0.8*U + np.random.normal(0, 0.5, n)

# Outcome: Y = 3 + 2*X + U + ε
# True causal effect of X on Y is β = 2
# U confounds the relationship
epsilon = np.random.normal(0, 1, n)
Y = 3 + 2*X + U + epsilon

data_iv = pd.DataFrame({'Y': Y, 'X': X, 'Z': Z})

# OLS (biased due to endogeneity through U)
model_ols = ols('Y ~ X', data=data_iv).fit()

# IV estimator manually computed
cov_zy = np.cov(Z, Y)[0, 1]
cov_zx = np.cov(Z, X)[0, 1]
beta_iv_manual = cov_zy / cov_zx

print("="*70)
print("SIMPLE IV ESTIMATOR: MANUAL CALCULATION")
print("="*70)
print(f"\nTrue causal effect (β): 2.0")
print(f"\nOLS estimate (BIASED):")
print(f"  β̂_OLS = {model_ols.params['X']:.4f}")
print(f"  Bias = {model_ols.params['X'] - 2:.4f} (upward bias due to U)")

print(f"\nIV estimator (manual):")
print(f"  Cov(Z, Y) = {cov_zy:.4f}")
print(f"  Cov(Z, X) = {cov_zx:.4f}")
print(f"  β̂^IV = Cov(Z,Y) / Cov(Z,X) = {beta_iv_manual:.4f}")
print(f"  ✓ Close to true value (β = 2.0)")

## 9.4 — Two-Stage Least Squares (2SLS)

**2SLS** extends IV to multiple regressors (some endogenous, some exogenous) and multiple instruments.

**Stage 1**: Regress each endogenous regressor on all instruments and exogenous variables.
   X̂ = Predicted value from Stage 1

**Stage 2**: Regress the outcome on X̂ (and exogenous variables).
   β̂^2SLS uses only the exogenous variation in X

In [ ]:
# Manual 2SLS: Step by step

# Stage 1: Predict X from Z
model_stage1 = ols('X ~ Z', data=data_iv).fit()
X_hat = model_stage1.fittedvalues

print("="*70)
print("TWO-STAGE LEAST SQUARES (2SLS) — MANUAL")
print("="*70)

print(f"\nSTAGE 1: Predict X from instrument Z")
print(f"  X̂ = {model_stage1.params['Intercept']:.4f} + {model_stage1.params['Z']:.4f}*Z")
print(f"  R² = {model_stage1.rsquared:.4f}")

# Stage 2: Regress Y on X_hat
data_iv['X_hat'] = X_hat
model_stage2 = ols('Y ~ X_hat', data=data_iv).fit()

print(f"\nSTAGE 2: Regress Y on predicted X̂")
print(f"  Ŷ = {model_stage2.params['Intercept']:.4f} + {model_stage2.params['X_hat']:.4f}*X̂")
print(f"\nManual 2SLS estimate: β̂₁^2SLS = {model_stage2.params['X_hat']:.4f}")
print(f"Standard error: SE = {model_stage2.bse['X_hat']:.4f}")
print(f"\nNote: Stage 2 SEs are INCORRECT because X̂ was estimated.")
print(f"We need proper 2SLS software that adjusts the SEs.")

In [ ]:
# Proper 2SLS using linearmodels.iv.IV2SLS
# Formula syntax: 'Y ~ 1 + [X ~ Z]'
# Inside brackets: [endogenous ~ instruments]

# Prepare data with appropriate types
data_iv_proper = data_iv[['Y', 'X', 'Z']].copy()

# Fit 2SLS model
model_2sls = IV2SLS.from_formula('Y ~ 1 + [X ~ Z]', data=data_iv_proper).fit()

print("="*70)
print("TWO-STAGE LEAST SQUARES (2SLS) — PROPER IMPLEMENTATION")
print("="*70)
print(model_2sls.summary)

In [ ]:
# Compare OLS, Manual 2SLS, and Proper 2SLS side by side

comparison = pd.DataFrame({
    'Estimator': ['OLS (biased)', 'IV (manual)', 'Manual 2SLS', 'Proper 2SLS'],
    'β̂': [
        model_ols.params['X'],
        beta_iv_manual,
        model_stage2.params['X_hat'],
        model_2sls.params['X']
    ],
    'SE': [
        model_ols.bse['X'],
        np.nan,  # Manual IV doesn't give SEs easily
        model_stage2.bse['X_hat'],  # Biased SEs
        model_2sls.std_errors['X']
    ]
})

print("\n" + "="*70)
print("COMPARISON: OLS vs IV vs 2SLS")
print("="*70)
print(comparison.to_string(index=False))
print(f"\nTrue causal effect: β = 2.0")
print(f"OLS is biased upward due to confounder U.")
print(f"2SLS uses exogenous variation in X (via Z) to estimate unbiased effect.")

## 9.5 — Testing Instrument Validity

Three key validity checks for instruments:

1. **Relevance**: Is the instrument correlated with the endogenous regressor?
   - Check: First-stage F-statistic (F > 10 is rule of thumb)
   
2. **Exogeneity**: Is the instrument uncorrelated with the error?
   - Check: Sargan/Hansen J-test (over-identifying restrictions)
   - Only testable with multiple instruments
   
3. **Exclusion**: Does the instrument affect Y only through X?
   - Check: Theoretical argument (cannot be tested statistically)

In [ ]:
# Extract first-stage F-statistic from 2SLS output

print("="*70)
print("INSTRUMENT VALIDITY TEST: FIRST-STAGE F-STATISTIC")
print("="*70)

# Get first-stage results from IV2SLS
first_stage = model_2sls.first_stage

print(f"\nFirst-stage regression: X ~ 1 + Z")
print(first_stage.summary)

print(f"\nFirst-stage F-statistic: {first_stage.f_statistic.stat:.4f}")
print(f"P-value: {first_stage.f_statistic.pval:.4f}")
print(f"\nRule of thumb: F > 10 indicates strong instrument relevance.")

if first_stage.f_statistic.stat > 10:
    print(f"✓ Instrument is STRONG (F = {first_stage.f_statistic.stat:.2f} > 10)")
else:
    print(f"✗ Instrument may be WEAK (F = {first_stage.f_statistic.stat:.2f} ≤ 10)")

In [ ]:
# Hausman test: compare OLS vs IV estimates
# If OLS and IV estimates differ significantly, OLS is likely biased (endogeneity)

from scipy.stats import chi2

# Compute difference in estimates
beta_diff = model_2sls.params['X'] - model_ols.params['X']

# For illustration, use simplified Hausman test
print("="*70)
print("HAUSMAN TEST: OLS vs IV")
print("="*70)
print(f"\nOLS estimate: {model_ols.params['X']:.4f}")
print(f"IV estimate:  {model_2sls.params['X']:.4f}")
print(f"Difference:   {beta_diff:.4f}")

if abs(beta_diff) > 0.3:
    print(f"\nThe estimates differ substantially.")
    print(f"This suggests OLS is biased due to endogeneity.")
else:
    print(f"\nThe estimates are similar.")
    print(f"Less evidence of endogeneity bias.")

## 9.6 — Weak Instruments Demonstration

When an instrument is **weakly correlated with the endogenous regressor**, the IV estimator becomes unreliable:

- First-stage F-statistic is low (< 10)
- Standard errors are inflated
- Confidence intervals become very wide
- Bias can be as large as or larger than OLS

Weak instruments are a common problem in practice.

In [ ]:
# Demonstrate weak instruments
# Create a weak instrument with very low correlation with X

n = 500
np.random.seed(42)

# Weak instrument: very low correlation with X
Z_weak = np.random.normal(0, 1, n)

# Confounder
U = np.random.normal(0, 1, n)

# X depends weakly on Z_weak
X_weak = 1 + 0.05*Z_weak + 0.8*U + np.random.normal(0, 0.5, n)  # π₁ = 0.05 (weak!)

# Y
epsilon = np.random.normal(0, 1, n)
Y_weak = 3 + 2*X_weak + U + epsilon

data_weak = pd.DataFrame({'Y': Y_weak, 'X': X_weak, 'Z': Z_weak})

# Fit 2SLS with weak instrument
model_2sls_weak = IV2SLS.from_formula('Y ~ 1 + [X ~ Z]', data=data_weak).fit()

print("="*70)
print("WEAK INSTRUMENTS DEMONSTRATION")
print("="*70)

print(f"\nStrong instrument (previous example):")
print(f"  First-stage F: {first_stage.f_statistic.stat:.4f}")
print(f"  2SLS estimate: {model_2sls.params['X']:.4f}")
print(f"  2SLS SE:       {model_2sls.std_errors['X']:.4f}")

first_stage_weak = model_2sls_weak.first_stage

print(f"\nWeak instrument (this example):")
print(f"  First-stage F: {first_stage_weak.f_statistic.stat:.4f}")
print(f"  2SLS estimate: {model_2sls_weak.params['X']:.4f}")
print(f"  2SLS SE:       {model_2sls_weak.std_errors['X']:.4f}")

print(f"\nComparison:")
print(f"  SE ratio (weak/strong): {model_2sls_weak.std_errors['X'] / model_2sls.std_errors['X']:.2f}x")
print(f"  Weak instrument produces much larger SEs and unreliable estimates.")

## 9.7 — Practical Example: Returns to Education

A classic application in labor economics: estimating the causal effect of education on wages.

**Endogeneity problem**: Ability and motivation affect both education and wages.

**Instrument**: Quarter of birth (used in Angrist & Krueger 1991)
- Affects schooling attainment (due to compulsory schooling age laws)
- Uncorrelated with ability (random)

This is a simplified stylised version of that analysis.

In [ ]:
# Simulated education-wages example with quarter of birth as IV

n = 1000
np.random.seed(42)

# Ability (unobserved confounder)
ability = np.random.normal(0, 1, n)

# Quarter of birth (instrument): 1, 2, 3, or 4
quarter = np.random.choice([1, 2, 3, 4], n)

# Schooling depends on quarter of birth (due to compulsory schooling age)
# Educational attainment affected by quarter + ability
schooling = 12 + 0.15*(quarter - 2.5) + 0.5*ability + np.random.normal(0, 0.8, n)

# Wages affected by schooling and ability
# True causal return to education: 0.08 (8% wage increase per year of schooling)
wages = 2 + 0.08*schooling + 0.5*ability + np.random.normal(0, 0.3, n)

data_education = pd.DataFrame({
    'wages': wages,
    'schooling': schooling,
    'quarter': quarter
})

# OLS estimate (biased upward due to ability)
model_ols_edu = ols('wages ~ schooling', data=data_education).fit()

# IV estimate using quarter as instrument
model_iv_edu = IV2SLS.from_formula('wages ~ 1 + [schooling ~ quarter]', 
                                   data=data_education).fit()

print("="*70)
print("PRACTICAL EXAMPLE: RETURNS TO EDUCATION")
print("="*70)
print(f"\nTrue causal return to education: 0.08 (8%)")
print(f"\nOLS estimate (biased):")
print(f"  β̂_OLS = {model_ols_edu.params['schooling']:.4f}")
print(f"  SE = {model_ols_edu.bse['schooling']:.4f}")
print(f"  ✗ Biased upward due to unobserved ability")

print(f"\nIV estimate (quarter of birth as instrument):")
print(f"  β̂^IV = {model_iv_edu.params['schooling']:.4f}")
print(f"  SE = {model_iv_edu.std_errors['schooling']:.4f}")
print(f"  ✓ Closer to true causal effect")

first_stage_edu = model_iv_edu.first_stage
print(f"\nFirst-stage F-statistic: {first_stage_edu.f_statistic.stat:.4f}")
if first_stage_edu.f_statistic.stat > 10:
    print(f"  ✓ Strong instrument")
else:
    print(f"  ✗ Weak instrument")

In [ ]:
# Visualize IV estimate vs OLS for education example

fig, ax = plt.subplots(figsize=(10, 6))

estimates = [model_ols_edu.params['schooling'], model_iv_edu.params['schooling']]
ses = [model_ols_edu.bse['schooling'], model_iv_edu.std_errors['schooling']]
methods = ['OLS\n(Biased)', 'IV (Quarter of Birth)\n(Unbiased)']

x_pos = np.arange(len(estimates))
colors = ['red', 'green']

ax.bar(x_pos, estimates, yerr=1.96*np.array(ses), capsize=10, 
       color=colors, alpha=0.6, edgecolor='black', linewidth=1.5)
ax.axhline(y=0.08, color='blue', linestyle='--', linewidth=2, label='True Effect (0.08)')
ax.set_ylabel('Return to Education Coefficient', fontsize=12)
ax.set_title('Education-Wages: OLS vs IV Estimates', fontsize=13, fontweight='bold')
ax.set_xticks(x_pos)
ax.set_xticklabels(methods)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, axis='y')
ax.set_ylim(0, 0.15)
plt.tight_layout()
plt.show()

## 9.8 — Summary: When to Use IV

Use **instrumental variables** when:

1. You have a specific endogeneity problem (omitted confounder, reverse causality, measurement error)
2. You have a valid instrument that satisfies relevance and exogeneity
3. Your research question is fundamentally causal

**Advantages:**
- Provides unbiased estimates when endogeneity is present
- Allows causal interpretation under exclusion restriction

**Disadvantages:**
- Relies on untestable exclusion restriction
- Weak instruments produce unreliable estimates
- Standard errors typically larger than OLS (less precise estimates)
- Difficult to find valid instruments in practice

In [ ]:
# Summary table: Properties of different estimators

summary_comparison = pd.DataFrame({
    'Property': ['Unbiased?', 'Consistent?', 'Efficient?', 'Testable?', 'Main Assumption'],
    'OLS': ['Yes (no endo)', 'Yes (no endo)', 'Yes (BLUE)', 'N/A', 'Exogeneity'],
    'IV/2SLS': ['Yes', 'Yes', 'No (less)', 'Partial', 'Relevance+Exogeneity']
})

print('\n' + '='*80)
print('ESTIMATOR COMPARISON SUMMARY')
print('='*80)
print(summary_comparison.to_string(index=False))
print('\nNote: Partial testability means we can test relevance (F-test) but not')
print('exogeneity with a single instrument.')